# Big Data: Data Unions and Joins Pipeline
## Lab 04 
Giselle Tino Flores

In [23]:
from Giselle_Module.spark_utils import SparkUtils
spark_url = "spark://spark-master:7077"
app_name = "Lab 04"
su = SparkUtils(spark_url, app_name)
su


<SparkContext master=local[*] appName=Spark SQL example>

In [24]:
base_path = "/opt/spark/work-dir/data/car_service"

In [25]:
from pyspark.sql.functions import col

spark = su._spark

In [26]:
# Agencies Dataframe

agencies_schema = SparkUtils.generate_schema([
    ("agency_id", "int"),
    ("agency_info", "string"),
])

agencies_df = (
    spark.read
      .option("header", True)
      .schema(agencies_schema)
      .csv(f"{base_path}/agencies/agencies.csv")
)

In [27]:
# Brands Dataframe

brands_schema = SparkUtils.generate_schema([
    ("brand_id", "int"),
    ("brand_info", "string"),
])

brands_df = (
    spark.read.option("header", True)
      .schema(brands_schema)
      .csv(f"{base_path}/brands/brands.csv")
)



In [28]:
# Cars Dataframe

cars_schema = SparkUtils.generate_schema([
    ("car_id", "int"),
    ("car_info", "string"),
])

cars_df = (
    spark.read.option("header", True)
      .schema(cars_schema)
      .csv(f"{base_path}/cars/cars.csv")
)


In [29]:
# Customers Dataframe

customers_schema = SparkUtils.generate_schema([
    ("customer_id", "int"),
    ("customer_info", "string"),
])

customers_df = (
    spark.read.option("header", True)
      .schema(customers_schema)
      .csv(f"{base_path}/customers/customers.csv")
)

In [30]:
# Rentals Dataframe

rentals_schema = SparkUtils.generate_schema([
    ("rental_id", "long"),
    ("rental_info", "string"),
])

rentals_df = (
    spark.read
      .option("header", True) 
      .schema(rentals_schema)
      .csv(f"{base_path}/rentals/")   # lee todos los part.csv
)

In [31]:
from pyspark.sql.functions import get_json_object, col

# Final Dataframe

df_join = rentals_df \
    .join(cars_df,get_json_object(col("rental_info"),"$.car_id")==col("car_id")) \
    .join(customers_df,get_json_object(col("rental_info"),"$.customer_id")==col("customer_id")) \
    .join(agencies_df,get_json_object(col("rental_info"),"$.agency_id")==col("agency_id"))

df = df_join.select(
    col("rental_id"),get_json_object(col("car_info"),"$.car_name").alias("car_name"),
        get_json_object(col("agency_info"),"$.agency_name").alias("agency_name"),
        get_json_object(col("customer_info"),"$.customer_name").alias("customer_name"),
)
df.show()

[Stage 11:>                                                         (0 + 1) / 1]

+---------+--------------------+-------------+---------------+
|rental_id|            car_name|  agency_name|  customer_name|
+---------+--------------------+-------------+---------------+
|    11891|Wallace-Carlson M...|  NYC Rentals| Margaret Jones|
|    11892|Grimes-Green Model 8|LA Car Rental|Albert Williams|
|    11893|Stewart-Allen Mod...|      SF Cars|  Caleb Fleming|
|    11894|  Campos PLC Model 4|  NYC Rentals|  Andrew Butler|
|    11895|  Wagner LLC Model 1|      SF Cars|  Kristin Potts|
|    11896|Jones, Jefferson ...|LA Car Rental|   Jeremy Parks|
|    11897|Lopez and Sons Mo...| Zapopan Auto|    Terry Wells|
|    11898| Salazar Ltd Model 8|      SF Cars|  Marc Williams|
|    11899|Villanueva PLC Mo...|LA Car Rental| Danny Williams|
|    11900|Faulkner-Howard M...|      SF Cars| Eric Owens PhD|
|    11901|Faulkner-Howard M...|  NYC Rentals|    Laura Perry|
|    11902|Faulkner-Howard M...|  NYC Rentals|     Paul Brown|
|    11903|Atkinson Ltd Mode...| Zapopan Auto|Alexa Her

In [32]:
su._spark.stop()